We are building a Customer Support Chatbot API using PydanticAI, FastAPI, and Ollama (local LLM). The chatbot takes a customer's message, uses an AI agent to analyze it, and returns a structured support ticket with a category, urgency level, suggested reply, and escalation decision — every single time, in a guaranteed format. All tickets are stored in a SQLite database and exposed through clean REST API endpoints built with FastAPI.

Installing dependencies

In [1]:
!pip install pydantic-ai[openai] fastapi uvicorn python-dotenv httpx aiosqlite nest-asyncio rich

In [2]:
import ollama
ollama.list()

ListResponse(models=[Model(model='llava:latest', modified_at=datetime.datetime(2026, 5, 20, 17, 48, 2, 43207, tzinfo=TzInfo(19800)), digest='8dd30f6b0cb19f555f2c7a7ebda861449ea2cc76bf1f44e262931f45fc81d081', size=4733363377, details=ModelDetails(parent_model='', format='gguf', family='llama', families=['llama', 'clip'], parameter_size='7B', quantization_level='Q4_0')), Model(model='llama3.2:latest', modified_at=datetime.datetime(2026, 5, 20, 17, 41, 31, 664809, tzinfo=TzInfo(19800)), digest='a80c4f17acd55265feec403c7aef86be0c25983ab279d83f3bcd3abbcb5b8b72', size=2019393189, details=ModelDetails(parent_model='', format='gguf', family='llama', families=['llama'], parameter_size='3.2B', quantization_level='Q4_K_M')), Model(model='qwen2.5:3b', modified_at=datetime.datetime(2026, 5, 19, 12, 42, 27, 537835, tzinfo=TzInfo(19800)), digest='357c53fb659c5076de1d65ccb0b397446227b71a42be9d1603d46168015c9e4b', size=1929912432, details=ModelDetails(parent_model='', format='gguf', family='qwen2', fam

In [3]:
import os
import uuid
import asyncio
import threading
import aiosqlite
import nest_asyncio
from datetime import datetime
from enum import Enum
from typing import Optional

from rich import print as rprint

from pydantic import BaseModel, Field
from pydantic_ai import Agent
from pydantic_ai.models.ollama import OllamaModel
from openai import AsyncOpenAI
from pydantic_ai.providers.ollama import OllamaProvider

import httpx
import uvicorn
from fastapi import FastAPI, HTTPException

# ── Jupyter fix ──────────────────────────────────────
nest_asyncio.apply()

# ── Config directly in notebook ───────────────────────
OLLAMA_BASE_URL = "http://localhost:11434/v1"
MODEL_NAME      = "llava"
DB_PATH         = "support_tickets.db"

print("✅ Imports done")
print(f"✅ Ollama URL  : {OLLAMA_BASE_URL}")
print(f"✅ Model       : {MODEL_NAME}")
print(f"✅ Database    : {DB_PATH}")

✅ Imports done
✅ Ollama URL  : http://localhost:11434/v1
✅ Model       : llava
✅ Database    : support_tickets.db


In [4]:
# ── Enums — fixed choices the AI must pick from ───────
class Category(str, Enum):
    billing   = "billing"
    technical = "technical"
    shipping  = "shipping"
    general   = "general"
    refund    = "refund"

class Urgency(str, Enum):
    low    = "low"
    medium = "medium"
    high   = "high"

# ── Main ticket model — what the AI must return ───────
class SupportTicket(BaseModel):
    ticket_id       : str      = Field(default_factory=lambda: f"TKT-{uuid.uuid4().hex[:6].upper()}")
    category        : Category
    urgency         : Urgency
    suggested_reply : str      = Field(description="A helpful reply to send to the customer")
    escalate        : bool     = Field(description="True if this needs a human agent")
    created_at      : str      = Field(default_factory=lambda: datetime.now().strftime("%Y-%m-%d %H:%M:%S"))

# ── What the customer sends in ────────────────────────
class CustomerMessage(BaseModel):
    customer_id : str
    message     : str

# ── What our API sends back ───────────────────────────
class APIResponse(BaseModel):
    success : bool
    ticket  : SupportTicket

# ── Quick check ───────────────────────────────────────
rprint("[bold green]✅ Models defined successfully[/bold green]")
rprint("[bold yellow]📋 SupportTicket fields:[/bold yellow]")
rprint(SupportTicket.model_fields)

✅ Models defined successfully

📋 SupportTicket fields:

{
    'ticket_id': FieldInfo(annotation=str, required=False, default_factory=<lambda>),
    'category': FieldInfo(annotation=Category, required=True),
    'urgency': FieldInfo(annotation=Urgency, required=True),
    'suggested_reply': FieldInfo(
        annotation=str,
        required=True,
        description='A helpful reply to send to the customer'
    ),
    'escalate': FieldInfo(annotation=bool, required=True, description='True if this needs a human agent'),
    'created_at': FieldInfo(annotation=str, required=False, default_factory=<lambda>)
}

Note:
Category(str, Enum)
  ├── defines FIXED categories the AI can pick from
  ├── billing | technical | shipping | general | refund
  └── AI cannot make up its own category like "other stuff"
      it MUST pick one of these 5 — that's the power of Enum

Urgency(str, Enum)
  ├── same idea — fixed urgency levels
  └── low | medium | high — nothing else allowed

SupportTicket(BaseModel)        ← the most important model
  ├── ticket_id
  │     └── auto-generated unique ID like TKT-A1B2C3
  │         uuid4 generates random ID, hex[:6] takes
  │         first 6 characters, upper() makes it caps
  │
  ├── category   → must be one of Category enum values
  ├── urgency    → must be one of Urgency enum values
  │
  ├── suggested_reply
  │     └── the actual reply text to send to customer
  │         Field(description=...) tells the AI what
  │         this field is for
  │
  ├── escalate
  │     └── True  = send to human agent
  │         False = chatbot can handle it
  │
  └── created_at
        └── auto-generated timestamp when ticket is made

CustomerMessage(BaseModel)
  ├── customer_id  →  who is sending the message
  └── message      →  what they are saying

APIResponse(BaseModel)
  ├── success  →  did it work? True or False
  └── ticket   →  the full SupportTicket object

In [5]:
# ── Create database and tickets table ─────────────────
async def init_db():
    async with aiosqlite.connect(DB_PATH) as db:
        await db.execute("""
            CREATE TABLE IF NOT EXISTS tickets (
                ticket_id       TEXT PRIMARY KEY,
                customer_id     TEXT NOT NULL,
                message         TEXT NOT NULL,
                category        TEXT NOT NULL,
                urgency         TEXT NOT NULL,
                suggested_reply TEXT NOT NULL,
                escalate        INTEGER NOT NULL,
                created_at      TEXT NOT NULL
            )
        """)
        await db.commit()
        print("✅ Database initialized")
        print(f"✅ Table 'tickets' ready at: {DB_PATH}")

# ── Save a ticket to database ─────────────────────────
async def save_ticket(customer_id: str, message: str, ticket: SupportTicket):
    async with aiosqlite.connect(DB_PATH) as db:
        await db.execute("""
            INSERT INTO tickets (
                ticket_id, customer_id, message, category,
                urgency, suggested_reply, escalate, created_at
            ) VALUES (?, ?, ?, ?, ?, ?, ?, ?)
        """, (
            ticket.ticket_id,
            customer_id,
            message,
            ticket.category.value,
            ticket.urgency.value,
            ticket.suggested_reply,
            1 if ticket.escalate else 0,
            ticket.created_at
        ))
        await db.commit()
        print(f"✅ Ticket {ticket.ticket_id} saved to database")

# ── Get a single ticket by ID ─────────────────────────
async def get_ticket(ticket_id: str):
    async with aiosqlite.connect(DB_PATH) as db:
        db.row_factory = aiosqlite.Row
        async with db.execute(
            "SELECT * FROM tickets WHERE ticket_id = ?", (ticket_id,)
        ) as cursor:
            row = await cursor.fetchone()
            return dict(row) if row else None

# ── Get all tickets ───────────────────────────────────
async def get_all_tickets():
    async with aiosqlite.connect(DB_PATH) as db:
        db.row_factory = aiosqlite.Row
        async with db.execute(
            "SELECT * FROM tickets ORDER BY created_at DESC"
        ) as cursor:
            rows = await cursor.fetchall()
            return [dict(row) for row in rows]

# ── Run the init ──────────────────────────────────────
await init_db()

✅ Database initialized
✅ Table 'tickets' ready at: support_tickets.db


init_db()
  ├── connects to support_tickets.db (creates it if not exists)
  ├── CREATE TABLE IF NOT EXISTS
  │     └── only creates table if it doesn't already exist
  │         so running this cell twice won't break anything
  └── defines 8 columns to store everything about a ticket

TABLE COLUMNS:
  ├── ticket_id       → unique ID like TKT-A1B2C3 (PRIMARY KEY)
  ├── customer_id     → who sent the message
  ├── message         → the original customer message
  ├── category        → billing / technical / shipping etc.
  ├── urgency         → low / medium / high
  ├── suggested_reply → the AI's reply text
  ├── escalate        → 0 or 1 (SQLite has no bool, so we use int)
  └── created_at      → timestamp

save_ticket()
  ├── takes customer_id, message, and a SupportTicket object
  ├── INSERT INTO tickets — saves one row
  ├── ticket.category.value
  │     └── .value gets the string out of the Enum
  │         Category.billing → "billing"
  └── 1 if ticket.escalate else 0
        └── converts Python bool to SQLite integer
            True  → 1
            False → 0

get_ticket()
  ├── finds ONE ticket by its ID
  ├── db.row_factory = aiosqlite.Row
  │     └── this makes the result behave like a dictionary
  │         so we can do row["ticket_id"] instead of row[0]
  └── returns dict(row) or None if not found

get_all_tickets()
  ├── fetches ALL tickets
  ├── ORDER BY created_at DESC
  │     └── newest tickets come first
  └── returns a list of dictionaries

In [6]:
# ── Define the model using OllamaModel + OllamaProvider ──
model = OllamaModel(
    model_name = MODEL_NAME,
    provider   = OllamaProvider(base_url = OLLAMA_BASE_URL)
)

# ── Build the Agent ───────────────────────────────────
agent = Agent(
    model         = model,
    output_type   = SupportTicket,        # ← was result_type, now output_type
    system_prompt = """
    You are a professional customer support agent.
    
    Your job is to analyze customer messages and return a structured response.
    
    Follow these rules strictly:
    
    1. CATEGORY — pick the most relevant one:
       - billing   → payment, invoice, charge, refund related
       - technical → app, software, login, error, bug related
       - shipping  → delivery, order, tracking, package related
       - refund    → return, refund, money back related
       - general   → anything else that doesn't fit above
    
    2. URGENCY — decide based on impact:
       - high   → account blocked, double charged, data loss, cannot access
       - medium → feature not working, order delayed, partial issue
       - low    → general question, feedback, minor inconvenience
    
    3. SUGGESTED REPLY — write a professional, empathetic reply
       that directly addresses the customer's problem.
       Keep it under 3 sentences.
    
    4. ESCALATE — set to true if:
       - urgency is high
       - customer mentions legal action
       - customer is extremely frustrated
       - issue cannot be resolved automatically
    """
)

rprint("[bold green]✅ OllamaProvider     : [/bold green]", OLLAMA_BASE_URL)
rprint("[bold green]✅ Model loaded       : [/bold green]", MODEL_NAME)
rprint("[bold green]✅ Agent built        [/bold green]")

✅ OllamaProvider     :  http://localhost:11434/v1

✅ Model loaded       :  llava

✅ Agent built        

In [ ]:
# ── Test message ──────────────────────────────────────
test_message = "I was charged twice for my order and now my account is blocked!"

rprint("[bold yellow]📨 Customer Message:[/bold yellow]", test_message)
rprint("[bold yellow]⏳ Agent thinking...[/bold yellow]")

# ── Run the agent ─────────────────────────────────────
result = await agent.run(test_message)

# ── Get the structured ticket ─────────────────────────
ticket = result.output

# ── Print the result ──────────────────────────────────
rprint("\n[bold green]✅ Structured Ticket Returned:[/bold green]")
rprint(f"  🎫 Ticket ID      : {ticket.ticket_id}")
rprint(f"  📂 Category       : {ticket.category.value}")
rprint(f"  🚨 Urgency        : {ticket.urgency.value}")
rprint(f"  💬 Suggested Reply: {ticket.suggested_reply}")
rprint(f"  👤 Escalate       : {ticket.escalate}")
rprint(f"  🕐 Created At     : {ticket.created_at}")

📨 Customer Message: I was charged twice for my order and now my account is blocked!

⏳ Agent thinking...